# Lecture 02: N-gram Language Models

CS40008.01 · Baojian Zhou · Fudan University · September 16, 2026

**Question:** How can a model assign a probability to a sentence, and how do we know whether one model is better than another?

This notebook follows the slides in order. Exercises E01–E03 match the slide IDs; P01–P03 are optional practices for after class (P02 and P03 keep the smoothing examples that the deck covers in one page). Everything runs offline with the Python standard library.

| Exercise | Slide | What you compute |
| --- | --- | --- |
| E01 | Toy example of training a bigram LM | Bigram MLE from a three-sentence corpus |
| E02 | Intrinsic evaluation | Average log-likelihood of test sentences |
| E03 | Perplexity: interpretation | Perplexity of a uniform model over digits |
| P01 | Sentence sampling | Sample sentences from the toy bigram model |
| P02 | Smoothing N-gram LMs in one page | Laplace probabilities and reconstituted counts (optional) |
| P03 | Smoothing N-gram LMs in one page | Check that Good–Turing estimates form a distribution (optional) |

## E01 · Toy example of training a bigram LM

The corpus and vocabulary from the slide. BOS marks the start of a sentence and EOS the end. The maximum likelihood estimate for a bigram is

$$p_\theta(w_i \mid w_{i-1}) = \frac{C(w_{i-1}w_i)}{C(w_{i-1})}.$$

Run the cell, then compare its output with your hand calculation.

In [ ]:
from collections import Counter
from fractions import Fraction

BOS, EOS = "BOS", "EOS"
TOY_CORPUS = ["I am Sam", "Sam I am", "I do not like eggs and ham"]


def tokenize(sentence):
    return [BOS] + sentence.split() + [EOS]


def count_ngrams(sentences):
    unigrams, bigrams = Counter(), Counter()
    for sentence in sentences:
        tokens = tokenize(sentence)
        unigrams.update(tokens)
        bigrams.update(zip(tokens, tokens[1:]))
    return unigrams, bigrams


UNIGRAMS, BIGRAMS = count_ngrams(TOY_CORPUS)


def bigram_mle(previous, word):
    """Maximum likelihood estimate p(word | previous) as an exact fraction."""
    history = UNIGRAMS[previous]
    if history == 0:
        raise KeyError(f"{previous!r} never appears as a history")
    return Fraction(BIGRAMS[(previous, word)], history)


for previous, word in [(BOS, "I"), (BOS, "Sam"), ("I", "am"), ("I", "do"), ("Sam", EOS), ("am", "Sam")]:
    estimate = bigram_mle(previous, word)
    print(f"p({word} | {previous}) = {estimate} = {float(estimate):.2f}")

**Check:** the slide lists $2/3$, $1/3$, $2/3$, $1/3$, $1/2$, $1/2$. Every column of the parameter matrix sums to one: verify it for the history `I`.

In [ ]:
words_after_I = {word for (previous, word) in BIGRAMS if previous == "I"}
print("Words after I:", sorted(words_after_I))
print("Sum:", sum(bigram_mle("I", word) for word in words_after_I))

## E02 · Intrinsic evaluation: propose a metric

A better model gives **higher probability to unseen test sentences**. The slide's metric is the average log-likelihood per sentence,

$$\frac{1}{|\mathcal{D}_{\text{test}}|} \sum_{\mathbf{w}\in\mathcal{D}_{\text{test}}} \log p_\theta(\mathbf{w}).$$

The second test sentence contains a bigram that never occurred in training. Watch what happens to its log-probability.

In [ ]:
import math


def sentence_log_prob(sentence, model=bigram_mle):
    """Natural-log probability of a sentence under a bigram model; -inf for an unseen bigram."""
    tokens = tokenize(sentence)
    total = 0.0
    for previous, word in zip(tokens, tokens[1:]):
        probability = model(previous, word)
        if probability == 0:
            return float("-inf")
        total += math.log(probability)
    return total


TEST_SENTENCES = ["I am Sam", "Sam do not like ham"]
for sentence in TEST_SENTENCES:
    print(f"log p({sentence!r}) = {sentence_log_prob(sentence):.4f}")

average_log_likelihood = sum(sentence_log_prob(s) for s in TEST_SENTENCES) / len(TEST_SENTENCES)
print("Average log-likelihood per sentence:", average_log_likelihood)

One zero makes the whole test set score $-\infty$: the motivation for smoothing in the third part of the lecture. Also note that longer sentences get lower log-probabilities, so per-token normalization is needed to compare test sets of different lengths. That is perplexity.

## E03 · Perplexity of a uniform model over digits

$$\mathrm{PPL}(s_{1:m}) = P_\theta(s_{1:m})^{-1/T} = \exp\left(-\frac{1}{T}\log P_\theta(s_{1:m})\right)$$

A model that guesses each of ten digits uniformly assigns $P = (1/10)^t$ to any digit string of length $t$. Predict the perplexity before running the cell.

In [ ]:
def perplexity(total_log_prob, token_count):
    """Perplexity from a natural-log probability and the number of predicted tokens."""
    return math.exp(-total_log_prob / token_count)


for length in [1, 5, 50]:
    log_prob = length * math.log(1 / 10)
    print(f"t = {length:>2}: perplexity = {perplexity(log_prob, length):.6f}")

digit_perplexity = perplexity(50 * math.log(1 / 10), 50)

**Check:** the answer is $10$ for every length, the effective branching factor. A vocabulary of size one gives perplexity $1$.

The toy bigram model can be scored the same way on a sentence it has seen. Perplexities are only comparable between models that share the same tokenization.

In [ ]:
seen = "I am Sam"
tokens = len(tokenize(seen)) - 1  # predicted tokens: everything after BOS
print(f"Perplexity of {seen!r} under the toy bigram model: {perplexity(sentence_log_prob(seen), tokens):.4f}")

## P02 · Optional: Laplace smoothing on the Berkeley Restaurant counts

The deck covers smoothing in one page. This practice works through the classic add-one example: bigram counts and unigram totals from Jurafsky and Martin, Chapter 3, with $|V| = 1446$.

$$P_{\text{Lap}}(w_i\mid w_{i-1})=\frac{C(w_{i-1}w_i)+1}{C(w_{i-1})+|V|}, \qquad C^{\ast}(w_{i-1}w_i) = P_{\text{Lap}}(w_i\mid w_{i-1})\cdot C(w_{i-1})$$

Compute $P_{\text{Lap}}(\text{want}\mid\text{i})$ by hand first, then run the cell and compare with the textbook tables.

In [ ]:
WORDS = ["i", "want", "to", "eat", "chinese", "food", "lunch", "spend"]
UNIGRAM_TOTALS = {"i": 2533, "want": 927, "to": 2417, "eat": 746,
                  "chinese": 158, "food": 1093, "lunch": 341, "spend": 278}
VOCAB_SIZE = 1446
BIGRAM_COUNTS = {
    "i":       [5, 827, 0, 9, 0, 0, 0, 2],
    "want":    [2, 0, 608, 1, 6, 6, 5, 1],
    "to":      [2, 0, 4, 686, 2, 0, 6, 211],
    "eat":     [0, 0, 2, 0, 16, 2, 42, 0],
    "chinese": [1, 0, 0, 0, 0, 82, 1, 0],
    "food":    [15, 0, 15, 0, 1, 4, 0, 0],
    "lunch":   [2, 0, 0, 0, 0, 1, 0, 0],
    "spend":   [1, 0, 1, 0, 0, 0, 0, 0],
}


def laplace_probability(previous, word):
    count = BIGRAM_COUNTS[previous][WORDS.index(word)]
    return (count + 1) / (UNIGRAM_TOTALS[previous] + VOCAB_SIZE)


def reconstituted_count(previous, word):
    return laplace_probability(previous, word) * UNIGRAM_TOTALS[previous]


def show_table(cell, title):
    print(title)
    print(f"{'':>8}" + "".join(f"{word:>10}" for word in WORDS))
    for previous in WORDS:
        print(f"{previous:>8}" + "".join(f"{cell(previous, word):>10.2g}" for word in WORDS))
    print()


laplace_table = {p: {w: laplace_probability(p, w) for w in WORDS} for p in WORDS}
reconstituted_table = {p: {w: reconstituted_count(p, w) for w in WORDS} for p in WORDS}
show_table(laplace_probability, "Laplace probabilities P_Lap(w_i | w_{i-1})")
show_table(reconstituted_count, "Reconstituted counts C*(w_{i-1} w_i)")
print(f"P_Lap(want | i) = {laplace_probability('i', 'want'):.4f}")
print(f"C*(i want) = {reconstituted_count('i', 'want'):.1f} (raw count 827)")
print(f"C*(want to) = {reconstituted_count('want', 'to'):.1f} (raw count 608)")

**Check:** $P_{\text{Lap}}(\text{want}\mid\text{i}) = 828/3979 = 0.208$. The reconstituted count of `i want` drops from 827 to 527 and `want to` from 608 to 238: add-one smoothing moves too much mass to the 1446 possible next words.

**Discussion:** which row changes most, and why does the size of $C(w_{i-1})$ relative to $|V|$ matter?

## P03 · Optional: Good–Turing, is $P^{\ast}_{\text{GT}}$ a distribution?

Text: **Sam I am I am Sam I do not eat** ($N = 10$ tokens). $N_r$ is the number of word types seen exactly $r$ times, and the vocabulary also contains three unseen words: Bob, Ben, you.

$$r^{\ast} = \frac{(r+1)N_{r+1}}{N_r}, \qquad P^{\ast}_{\text{GT}}(w) = \frac{r^{\ast}}{N}, \qquad P^{\ast}_{\text{GT}}(\text{unseen, total}) = \frac{N_1}{N}.$$

In [ ]:
GT_TEXT = "Sam I am I am Sam I do not eat"
UNSEEN = ["Bob", "Ben", "you"]

gt_counts = Counter(GT_TEXT.split())
N = sum(gt_counts.values())
frequency_of_frequency = Counter(gt_counts.values())
frequency_of_frequency[0] = len(UNSEEN)
print("Counts:", dict(gt_counts))
print("N_r:", {r: frequency_of_frequency[r] for r in sorted(frequency_of_frequency)})


def adjusted_count(r):
    return Fraction((r + 1) * frequency_of_frequency[r + 1], frequency_of_frequency[r])


good_turing = {}
for word in UNSEEN:
    good_turing[word] = Fraction(frequency_of_frequency[1], N) / frequency_of_frequency[0]
for word, r in gt_counts.items():
    good_turing[word] = adjusted_count(r) / N

for word, probability in sorted(good_turing.items(), key=lambda item: -item[1]):
    print(f"P*_GT({word:>4}) = {str(probability):>5} = {float(probability):.3f}")
good_turing_total = sum(good_turing.values())
print("Total:", good_turing_total)

**Check:** the total is exactly $1$, so the estimates form a distribution. But the most frequent word, `I`, receives probability $0$ because $N_4 = 0$. Practical Good–Turing smooths the $N_r$ values or applies the adjustment only for $r$ below a threshold; Kneser–Ney instead subtracts a fixed discount $d$ and interpolates.

## P01 · Sample sentences from the toy bigram model

Sampling from a bigram model: start from BOS, pick the next word from $p_\theta(\cdot \mid w_{t-1})$, and stop at EOS. This is the answer to the slide's question “What about the bigram case?” Run this before P02 and P03; it belongs to the sampling slide.

In [ ]:
import random


def sample_sentence(rng, max_length=20):
    tokens = [BOS]
    while tokens[-1] != EOS and len(tokens) < max_length:
        previous = tokens[-1]
        candidates = [(word, BIGRAMS[(prev, word)]) for (prev, word) in BIGRAMS if prev == previous]
        words, weights = zip(*sorted(candidates))
        tokens.append(rng.choices(words, weights=weights)[0])
    return " ".join(tokens[1:-1] if tokens[-1] == EOS else tokens[1:])


rng = random.Random(2026)
for _ in range(5):
    print(sample_sentence(rng))

Every sampled sentence reuses bigrams from the three training sentences, and the model can produce sentences it never saw, such as `I am Sam I am`. Change the seed and observe which continuations are possible after `I`.

## References

- Jurafsky and Martin, *Speech and Language Processing*, Chapter 3: [N-gram Language Models](https://web.stanford.edu/~jurafsky/slp3/3.pdf).
- Bengio, Ducharme, Vincent, and Jauvin (2003). *A Neural Probabilistic Language Model*. JMLR 3.
- Spring 2026 Lecture 02 slides: <https://baojian.github.io/llm-26/slides/lecture-02-slides/>.